# DQN Breakout Training (Colab)
Run seeds 1 and 2 here while seed 0 runs locally.

**Before running:** Go to Runtime → Change runtime type → Select **T4 GPU**

In [ ]:
# Install dependencies
!pip install torch torchvision gymnasium "gymnasium[atari]" ale-py autorom tensorboard matplotlib opencv-python-headless
!AutoROM --accept-license

In [ ]:
# Create project structure
!mkdir -p src configs results

In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/wrappers.py
import gymnasium as gym
import ale_py
import numpy as np
from collections import deque
import cv2

gym.register_envs(ale_py)


class NoopResetWrapper(gym.Wrapper):
    def __init__(self, env, noop_max=30):
        super().__init__(env)
        self.noop_max = noop_max
        self.noop_action = 0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        noops = np.random.randint(1, self.noop_max + 1)
        for _ in range(noops):
            obs, _, terminated, truncated, info = self.env.step(self.noop_action)
            if terminated or truncated:
                obs, info = self.env.reset(**kwargs)
        return obs, info


class FireResetWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        assert env.unwrapped.get_action_meanings()[1] == "FIRE"

    def reset(self, **kwargs):
        self.env.reset(**kwargs)
        obs, _, terminated, truncated, info = self.env.step(1)
        if terminated or truncated:
            obs, info = self.env.reset(**kwargs)
        return obs, info


class EpisodicLifeWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.lives = 0
        self.real_done = True

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.real_done = terminated or truncated
        lives = self.env.unwrapped.ale.lives()
        if 0 < lives < self.lives:
            terminated = True
        self.lives = lives
        return obs, reward, terminated, truncated, info

    def reset(self, **kwargs):
        if self.real_done:
            obs, info = self.env.reset(**kwargs)
        else:
            obs, _, _, _, info = self.env.step(0)
        self.lives = self.env.unwrapped.ale.lives()
        return obs, info


class MaxAndSkipWrapper(gym.Wrapper):
    def __init__(self, env, skip=4):
        super().__init__(env)
        self._skip = skip
        self._obs_buffer = np.zeros((2,) + env.observation_space.shape, dtype=np.uint8)

    def step(self, action):
        total_reward = 0.0
        terminated = truncated = False
        for i in range(self._skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            total_reward += reward
            if i == self._skip - 2:
                self._obs_buffer[0] = obs
            if i == self._skip - 1:
                self._obs_buffer[1] = obs
            if terminated or truncated:
                break
        max_frame = self._obs_buffer.max(axis=0)
        return max_frame, total_reward, terminated, truncated, info

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._obs_buffer[0] = obs
        self._obs_buffer[1] = obs
        return obs, info


class GrayscaleResizeWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(84, 84), dtype=np.uint8)

    def observation(self, obs):
        gray = np.dot(obs[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8)
        resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
        return resized


class FrameStackWrapper(gym.Wrapper):
    def __init__(self, env, n_frames=4):
        super().__init__(env)
        self.n_frames = n_frames
        self.frames = deque([], maxlen=n_frames)
        old_space = env.observation_space
        self.observation_space = gym.spaces.Box(
            low=0, high=255,
            shape=(n_frames, old_space.shape[0], old_space.shape[1]),
            dtype=np.uint8,
        )

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for _ in range(self.n_frames):
            self.frames.append(obs)
        return self._get_obs(), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.frames.append(obs)
        return self._get_obs(), reward, terminated, truncated, info

    def _get_obs(self):
        return np.array(self.frames)


class ClipRewardWrapper(gym.RewardWrapper):
    def reward(self, reward):
        return np.sign(reward)


def make_atari_env(env_id="ALE/Breakout-v5", render_mode=None):
    env = gym.make(env_id, render_mode=render_mode)
    env = NoopResetWrapper(env, noop_max=30)
    env = MaxAndSkipWrapper(env, skip=4)
    env = EpisodicLifeWrapper(env)
    if "FIRE" in env.unwrapped.get_action_meanings():
        env = FireResetWrapper(env)
    env = GrayscaleResizeWrapper(env)
    env = ClipRewardWrapper(env)
    env = FrameStackWrapper(env, n_frames=4)
    return env

In [ ]:
%%writefile src/replay_buffer.py
import numpy as np


class ReplayBuffer:
    def __init__(self, capacity, obs_shape):
        self.capacity = capacity
        self.idx = 0
        self.size = 0
        self.observations = np.zeros((capacity, *obs_shape), dtype=np.uint8)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_observations = np.zeros((capacity, *obs_shape), dtype=np.uint8)
        self.dones = np.zeros(capacity, dtype=np.bool_)

    def push(self, obs, action, reward, next_obs, done):
        self.observations[self.idx] = obs
        self.actions[self.idx] = action
        self.rewards[self.idx] = reward
        self.next_observations[self.idx] = next_obs
        self.dones[self.idx] = done
        self.idx = (self.idx + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = np.random.randint(0, self.size, size=batch_size)
        return (
            self.observations[indices],
            self.actions[indices],
            self.rewards[indices],
            self.next_observations[indices],
            self.dones[indices],
        )

    def __len__(self):
        return self.size

In [ ]:
%%writefile src/model.py
import torch
import torch.nn as nn


class DQN(nn.Module):
    def __init__(self, n_actions, in_channels=4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        x = x.float() / 255.0
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [ ]:
%%writefile src/dqn_agent.py
import torch
import torch.nn.functional as F
import numpy as np
from src.model import DQN
from src.replay_buffer import ReplayBuffer


class DQNAgent:
    def __init__(self, n_actions, obs_shape, config, device="cpu"):
        self.n_actions = n_actions
        self.device = device
        self.config = config
        self.policy_net = DQN(n_actions, in_channels=obs_shape[0]).to(device)
        self.target_net = DQN(n_actions, in_channels=obs_shape[0]).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer = torch.optim.Adam(self.policy_net.parameters(), lr=config["lr"])
        self.replay_buffer = ReplayBuffer(config["buffer_size"], obs_shape)
        self.epsilon_start = config["epsilon_start"]
        self.epsilon_end = config["epsilon_end"]
        self.epsilon_decay_steps = config["epsilon_decay_steps"]
        self.steps_done = 0

    @property
    def epsilon(self):
        frac = min(1.0, self.steps_done / self.epsilon_decay_steps)
        return self.epsilon_start + frac * (self.epsilon_end - self.epsilon_start)

    def select_action(self, obs):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        with torch.no_grad():
            obs_t = torch.tensor(obs, dtype=torch.uint8, device=self.device).unsqueeze(0)
            q_values = self.policy_net(obs_t)
            return q_values.argmax(dim=1).item()

    def store_transition(self, obs, action, reward, next_obs, done):
        self.replay_buffer.push(obs, action, reward, next_obs, done)
        self.steps_done += 1

    def train_step(self):
        if len(self.replay_buffer) < self.config["learning_starts"]:
            return None
        if self.steps_done % self.config["train_frequency"] != 0:
            return None
        batch_size = self.config["batch_size"]
        gamma = self.config["gamma"]
        obs, actions, rewards, next_obs, dones = self.replay_buffer.sample(batch_size)
        obs_t = torch.tensor(obs, dtype=torch.uint8, device=self.device)
        actions_t = torch.tensor(actions, dtype=torch.long, device=self.device)
        rewards_t = torch.tensor(rewards, dtype=torch.float32, device=self.device)
        next_obs_t = torch.tensor(next_obs, dtype=torch.uint8, device=self.device)
        dones_t = torch.tensor(dones, dtype=torch.float32, device=self.device)
        q_values = self.policy_net(obs_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q_values = self.target_net(next_obs_t).max(dim=1)[0]
            target = rewards_t + gamma * next_q_values * (1.0 - dones_t)
        loss = F.smooth_l1_loss(q_values, target)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimizer.step()
        return loss.item()

    def sync_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def save(self, path):
        torch.save({
            "policy_net": self.policy_net.state_dict(),
            "target_net": self.target_net.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "steps_done": self.steps_done,
        }, path)

    def load(self, path):
        checkpoint = torch.load(path, map_location=self.device)
        self.policy_net.load_state_dict(checkpoint["policy_net"])
        self.target_net.load_state_dict(checkpoint["target_net"])
        self.optimizer.load_state_dict(checkpoint["optimizer"])
        self.steps_done = checkpoint["steps_done"]

In [ ]:
%%writefile configs/breakout.json
{
    "env_id": "ALE/Breakout-v5",
    "total_steps": 5000000,
    "buffer_size": 100000,
    "batch_size": 32,
    "gamma": 0.99,
    "lr": 1e-4,
    "epsilon_start": 1.0,
    "epsilon_end": 0.1,
    "epsilon_decay_steps": 1000000,
    "learning_starts": 10000,
    "train_frequency": 4,
    "target_update_frequency": 10000,
    "eval_frequency": 50000,
    "eval_episodes": 10,
    "save_frequency": 500000,
    "log_frequency": 5000
}

In [ ]:
%%writefile train_atari.py
import argparse
import json
import os
import time
from collections import deque

import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter

from src.wrappers import make_atari_env
from src.dqn_agent import DQNAgent


def evaluate(agent, env_id, n_episodes=10):
    env = make_atari_env(env_id)
    rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        episode_reward = 0.0
        done = False
        while not done:
            with torch.no_grad():
                obs_t = torch.tensor(obs, dtype=torch.uint8, device=agent.device).unsqueeze(0)
                action = agent.policy_net(obs_t).argmax(dim=1).item()
            obs, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward
            done = terminated or truncated
        rewards.append(episode_reward)
    env.close()
    return np.mean(rewards), np.std(rewards)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, default="configs/breakout.json")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--device", type=str, default=None)
    args = parser.parse_args()

    with open(args.config) as f:
        config = json.load(f)

    if args.device:
        device = torch.device(args.device)
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Using device: {device}")

    seed = args.seed
    np.random.seed(seed)
    torch.manual_seed(seed)

    env_id = config["env_id"]
    env = make_atari_env(env_id)
    env.reset(seed=seed)

    n_actions = env.action_space.n
    obs_shape = env.observation_space.shape
    print(f"Environment: {env_id}, Actions: {n_actions}, Obs shape: {obs_shape}")

    agent = DQNAgent(n_actions, obs_shape, config, device=device)

    run_name = f"{env_id.split('/')[-1]}_seed{seed}_{int(time.time())}"
    log_dir = os.path.join("results", "runs", run_name)
    writer = SummaryWriter(log_dir)
    checkpoint_dir = os.path.join("results", "checkpoints", run_name)
    os.makedirs(checkpoint_dir, exist_ok=True)

    with open(os.path.join(log_dir, "config.json"), "w") as f:
        json.dump({**config, "seed": seed, "device": str(device)}, f, indent=2)

    obs, _ = env.reset()
    episode_reward = 0.0
    episode_num = 0
    recent_rewards = deque(maxlen=100)
    recent_losses = deque(maxlen=100)
    start_time = time.time()

    for step in range(1, config["total_steps"] + 1):
        action = agent.select_action(obs)
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        agent.store_transition(obs, action, reward, next_obs, terminated)
        loss = agent.train_step()
        if loss is not None:
            recent_losses.append(loss)
        if step % config["target_update_frequency"] == 0:
            agent.sync_target_network()
        episode_reward += reward
        if done:
            recent_rewards.append(episode_reward)
            writer.add_scalar("train/episode_reward", episode_reward, step)
            writer.add_scalar("train/epsilon", agent.epsilon, step)
            episode_num += 1
            episode_reward = 0.0
            obs, _ = env.reset()
        else:
            obs = next_obs
        if step % config["log_frequency"] == 0 and recent_rewards:
            elapsed = time.time() - start_time
            fps = step / elapsed
            avg_reward = np.mean(recent_rewards)
            avg_loss = np.mean(recent_losses) if recent_losses else 0.0
            writer.add_scalar("train/avg_reward_100", avg_reward, step)
            writer.add_scalar("train/avg_loss", avg_loss, step)
            writer.add_scalar("train/fps", fps, step)
            print(f"Step {step:>8d}/{config['total_steps']} | Ep {episode_num:>5d} | Avg100: {avg_reward:>7.2f} | Loss: {avg_loss:.4f} | Eps: {agent.epsilon:.3f} | FPS: {fps:.0f}")
        if step % config["eval_frequency"] == 0:
            eval_mean, eval_std = evaluate(agent, env_id, n_episodes=config["eval_episodes"])
            writer.add_scalar("eval/mean_reward", eval_mean, step)
            writer.add_scalar("eval/std_reward", eval_std, step)
            print(f"  [EVAL] Step {step}: {eval_mean:.2f} +/- {eval_std:.2f}")
        if step % config["save_frequency"] == 0:
            path = os.path.join(checkpoint_dir, f"checkpoint_{step}.pt")
            agent.save(path)
            print(f"  Saved checkpoint: {path}")

    agent.save(os.path.join(checkpoint_dir, "final.pt"))
    env.close()
    writer.close()
    print("Training complete.")


if __name__ == "__main__":
    main()

## Train Seed 1

In [ ]:
!python train_atari.py --config configs/breakout.json --seed 1 --device cuda

## Train Seed 2

In [ ]:
!python train_atari.py --config configs/breakout.json --seed 2 --device cuda

## Download Results
After training, download the results folder to your local machine.

In [ ]:
# Zip results for easy download
!zip -r results_seeds_1_2.zip results/

In [ ]:
from google.colab import files
files.download('results_seeds_1_2.zip')